# Train hourly LSTM

Single / double / bidir LSTM. Target: Appliances (Wh).

In [ ]:
FORCE_RETRAIN = False
window_sizes = [1, 4, 8, 12, 24, 48, 72, 168]
epochs = 50
batch_size = 32
test_ratio = 0.18

from pathlib import Path
ROOT = Path('..').resolve()
DATA_CSV = ROOT / 'outputs' / 'preprocess' / 'hourly' / 'data.csv'
SCALER_PKL = ROOT / 'outputs' / 'preprocess' / 'hourly' / 'scaler.pkl'
SAVE_PATH = ROOT / 'outputs' / 'train' / 'hourly'

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow

import os, pickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.models import Sequential, load_model


def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_true - y_pred)))

os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
scaled_df = pd.read_csv(DATA_CSV)
with open(SCALER_PKL, 'rb') as f:
    scaler = pickle.load(f)
feature_cols = list(scaled_df.columns)
target_idx = 0
n_features = len(feature_cols)

In [ ]:
arr = scaled_df[feature_cols].to_numpy(dtype=np.float32)
data = {}
for window in window_sizes:
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i:i + window])
        y.append(arr[i + window, target_idx])
    X, y = np.array(X), np.array(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_ratio, shuffle=False)
    data[f'win{window}'] = dict(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)

In [ ]:
stacks = ['single', 'double', 'bidir']
trained = {}
for stack in stacks:
    model_dir = os.path.join(SAVE_PATH, stack, 'models')
    os.makedirs(model_dir, exist_ok=True)
    trained[stack] = {}
    for window in window_sizes:
        mp = os.path.join(model_dir, f'win{window}.keras')
        d = data[f'win{window}']
        if os.path.exists(mp) and not FORCE_RETRAIN:
            trained[stack][window] = {'model': load_model(mp, custom_objects={'rmse': rmse})}
            print(stack, 'win'+str(window), 'loaded')
            continue
        model = Sequential()
        if stack == 'single':
            model.add(LSTM(64, input_shape=(window, n_features)))
        elif stack == 'double':
            model.add(LSTM(64, return_sequences=True, input_shape=(window, n_features)))
            model.add(LSTM(64))
        else:
            model.add(Bidirectional(LSTM(64), input_shape=(window, n_features)))
        model.add(Dropout(0.1))
        model.add(Dense(1))
        model.compile(optimizer='adam', loss=rmse, metrics=['mae'])
        model.fit(d['X_train'], d['y_train'], validation_data=(d['X_test'], d['y_test']),
                  epochs=epochs, batch_size=batch_size, verbose=1)
        model.save(mp)
        trained[stack][window] = {'model': model}
        print(stack, 'win'+str(window), 'saved')

In [ ]:
def to_wh(v):
    v = np.asarray(v).ravel()
    d = np.zeros((len(v), n_features))
    d[:, target_idx] = v
    return scaler.inverse_transform(d)[:, target_idx]

rows = []
for stack in stacks:
    for window in window_sizes:
        if window not in trained[stack]:
            continue
        m = trained[stack][window]['model']
        yt = data[f'win{window}']['y_test']
        yp = m.predict(data[f'win{window}']['X_test'], verbose=0).ravel()
        yt, yp = to_wh(yt), to_wh(yp)
        rows.append({'model': stack, 'window': window,
            'rmse_wh': float(np.sqrt(((yt-yp)**2).mean())),
            'mae_wh': float(np.abs(yt-yp).mean()),
            'r2': float(r2_score(yt, yp))})
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(os.path.join(SAVE_PATH, 'results_metrics.csv'), index=False)
print(metrics_df.sort_values('rmse_wh').head())

In [ ]:
pivot = metrics_df.pivot(index='window', columns='model', values='rmse_wh')
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd_r')
plt.title(f'RMSE Wh — {track}')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'rmse_heatmap.png'))
plt.show()